In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Load and clean the data
df = pd.read_csv("AB_NYC_2019.csv")
df.drop(['id', 'name', 'host_id', 'host_name'], axis=1, inplace=True)
df['reviews_per_month'].fillna(0, inplace=True)
df['last_review'] = pd.to_datetime(df['last_review'])
reference_date = df['last_review'].max()
df['days_since_last_review'] = (reference_date - df['last_review']).dt.days
df['days_since_last_review'].fillna(df['days_since_last_review'].max() + 1, inplace=True)
df.drop('last_review', axis=1, inplace=True)
df = df[df['price'] > 0].copy()


/var/folders/t9/2g61rzyn1tz5tkc9gshdkmyc0000gn/T/ipykernel_20193/452267584.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['reviews_per_month'].fillna(0, inplace=True)
/var/folders/t9/2g61rzyn1tz5tkc9gshdkmyc0000gn/T/ipykernel_20193/452267584.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alway

In [10]:
# Step 2: Feature engineering
df['multi_listings_host'] = df['calculated_host_listings_count'] > 1
df['reviews_bin'] = pd.cut(df['number_of_reviews'], bins=[-1, 0, 5, 20, 100, 1000],
                           labels=['0', '1-5', '6-20', '21-100', '100+'])
df['min_nights_bin'] = pd.cut(df['minimum_nights'], bins=[-1, 1, 7, 30, 365],
                               labels=['1', '2-7', '8-30', '31+'])
df['region_room'] = df['neighbourhood_group'] + '_' + df['room_type']

In [11]:
# Step 3: Encode categorical features
categorical = ['neighbourhood_group', 'neighbourhood', 'room_type', 'region_room', 'reviews_bin', 'min_nights_bin']
df_encoded = pd.get_dummies(df, columns=categorical, drop_first=True)

In [12]:
# Step 4: Log-transform the target
y = np.log(df_encoded['price'].values)
X = df_encoded.drop(columns=['price'])

In [13]:
# Step 5: Align X and y lengths
X = X.loc[~pd.isna(y)]
y = y[~pd.isna(y)]

In [14]:
# Step 6: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Step 7: Standardize numeric features
numeric_features = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
                    'reviews_per_month', 'calculated_host_listings_count', 'availability_365',
                    'days_since_last_review']
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# Step 8: Train LassoCV model
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)

# Step 9: Evaluate the model
y_pred = lasso.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(np.exp(y_test), np.exp(y_pred)))

print(f"Lasso Regression R^2: {r2:.3f}")
print(f"RMSE (in $): ${rmse:.2f}")
print(f"Optimal alpha: {lasso.alpha_:.5f}")

# Step 10: Display most influential features
coefs = pd.Series(lasso.coef_, index=X.columns)
top_features = coefs.abs().sort_values(ascending=False).head(10)
print("Top 10 Features:")
print(top_features)

# Step 11: Convert log predictions and actuals to actual price values
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

# Create a DataFrame to compare predicted and actual prices
price_comparison = pd.DataFrame({
    'Predicted Price ($)': np.round(y_pred_actual, 2),
    'Actual Price ($)': np.round(y_test_actual, 2)
})

# Print the first 10 comparisons
print("Predicted vs Actual Prices (first 10 rows):")
print(price_comparison.head(10))


Lasso Regression R^2: 0.543
RMSE (in $): $185.96
Optimal alpha: 0.00023
Top 10 Features:
room_type_Shared room                1.049478
room_type_Private room               0.677344
neighbourhood_group_Manhattan        0.495641
neighbourhood_Tribeca                0.491630
neighbourhood_Washington Heights     0.440851
neighbourhood_Inwood                 0.400587
neighbourhood_group_Staten Island    0.324726
min_nights_bin_8-30                  0.280644
neighbourhood_Williamsburg           0.279640
neighbourhood_Brooklyn Heights       0.274501
dtype: float64
Predicted vs Actual Prices (first 10 rows):
   Predicted Price ($)  Actual Price ($)
0                83.82              98.0
1                85.00              89.0
2               132.06              79.0
3                57.82              59.0
4               109.12              89.0
5                74.82              99.0
6               135.35             199.0
7                54.62              44.0
8                79.90 

In [8]:
def predict_price(sample_input, model, scaler, reference_columns, numeric_features):
    import pandas as pd
    import numpy as np

    # --- Step 1: Input DF ---
    df_input = pd.DataFrame([sample_input])

    # --- Step 2: Feature Engineering ---
    df_input['region_room'] = df_input['neighbourhood_group'] + "_" + df_input['room_type']
    df_input['multi_listings_host'] = 1 if sample_input.get("calculated_host_listings_count", 0) > 1 else 0

    reviews = sample_input.get("number_of_reviews", 0)
    if reviews == 0:
        df_input['reviews_bin'] = '0'
    elif 1 <= reviews <= 5:
        df_input['reviews_bin'] = '1-5'
    elif 6 <= reviews <= 20:
        df_input['reviews_bin'] = '6-20'
    else:
        df_input['reviews_bin'] = '21-100'  # match training bin

    nights = sample_input.get("minimum_nights", 0)
    if nights < 2:
        df_input['min_nights_bin'] = '1'
    elif 2 <= nights <= 7:
        df_input['min_nights_bin'] = '2-7'
    elif 8 <= nights <= 30:
        df_input['min_nights_bin'] = '8-30'
    else:
        df_input['min_nights_bin'] = '31+'

    # --- Step 3: One-hot encode ---
    df_encoded = pd.get_dummies(df_input, drop_first=True)

    # --- Step 4: Add missing columns
    for col in reference_columns:
        if col not in df_encoded.columns:
            df_encoded[col] = 0

    df_encoded = df_encoded[reference_columns]

    # --- Step 5: Scale numeric features safely ---
    for col in numeric_features:
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')
    df_encoded[numeric_features] = scaler.transform(df_encoded[numeric_features])

    # --- Step 6: Predict ---
    log_price = model.predict(df_encoded)[0]
    price = np.expm1(log_price)

    return round(price, 2)


In [11]:
# Sample input for prediction


sample_listing = {
    'neighbourhood_group': 'Manhattan',
    'neighbourhood': 'Harlem',
    'room_type': 'Private room',
    'minimum_nights': 3,
    'number_of_reviews': 20,
    'reviews_per_month': 1.5,
    'calculated_host_listings_count': 2,
    'availability_365': 180,
    'latitude': 40.8090,
    'longitude': -73.9440,
}

# Call the prediction function
predicted = predict_price(
    sample_input=sample_listing,
    model=lasso,  # Make sure this variable exists by rerunning its cell
    scaler=scaler,
    reference_columns=X_train.columns.tolist(),
    numeric_features=numeric_features
)

print(f"🎯 Predicted price for sample listing: ${predicted}")

True
🎯 Predicted price for sample listing: $140.44


/var/folders/t9/2g61rzyn1tz5tkc9gshdkmyc0000gn/T/ipykernel_4773/264933441.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_encoded[col] = 0
/var/folders/t9/2g61rzyn1tz5tkc9gshdkmyc0000gn/T/ipykernel_4773/264933441.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_encoded[col] = 0
/var/folders/t9/2g61rzyn1tz5tkc9gshdkmyc0000gn/T/ipykernel_4773/264933441.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi